In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp /content/drive/MyDrive/satellite/satellite_model_run.zip /content/.
!unzip /content/satellite_model_run.zip
!cp -r /content/content/runs/segment/train/weights/best.pt .

Archive:  /content/satellite_model_run.zip
   creating: content/runs/
   creating: content/runs/segment/
   creating: content/runs/segment/val/
  inflating: content/runs/segment/val/MaskR_curve.png  
  inflating: content/runs/segment/val/val_batch0_labels.jpg  
  inflating: content/runs/segment/val/val_batch1_pred.jpg  
  inflating: content/runs/segment/val/val_batch1_labels.jpg  
  inflating: content/runs/segment/val/confusion_matrix_normalized.png  
  inflating: content/runs/segment/val/MaskF1_curve.png  
  inflating: content/runs/segment/val/val_batch2_labels.jpg  
  inflating: content/runs/segment/val/BoxPR_curve.png  
  inflating: content/runs/segment/val/val_batch0_pred.jpg  
  inflating: content/runs/segment/val/BoxF1_curve.png  
  inflating: content/runs/segment/val/confusion_matrix.png  
  inflating: content/runs/segment/val/BoxP_curve.png  
  inflating: content/runs/segment/val/BoxR_curve.png  
  inflating: content/runs/segment/val/MaskPR_curve.png  
  inflating: content/runs

In [3]:
!pip install ultralytics streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 92.3 MB/s eta 0:00:00


In [4]:
import getpass
import os

if not os.environ.get("Ngrok_API_KEY"):
    os.environ["Ngrok_API_KEY"] = getpass.getpass("Enter API key for ngrok:\n")

Enter API key for ngrok:
··········


In [5]:
from ultralytics import YOLO
model = YOLO("best.pt")
model

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


YOLO(
  (model): SegmentationModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_runni

In [6]:
%%writefile app.py
import streamlit as st
import cv2
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import tempfile
import os

# ─────────────────────────────────────────────
# Page config
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="Urban Analysis",
    page_icon="🛰️",
    layout="wide",
    initial_sidebar_state="collapsed",
)

# ─────────────────────────────────────────────
# Custom CSS
# ─────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;500;600&display=swap');

:root {
    --bg: #0a0e1a;
    --surface: #111827;
    --surface2: #1a2235;
    --border: #1e2d45;
    --accent: #00d4ff;
    --accent2: #7c3aed;
    --green: #10b981;
    --orange: #f59e0b;
    --red: #ef4444;
    --text: #e2e8f0;
    --muted: #64748b;
}

html, body, .stApp {
    background-color: var(--bg) !important;
    font-family: 'DM Sans', sans-serif;
    color: var(--text);
}

/* Hide streamlit default elements */
#MainMenu, footer, header { visibility: hidden; }
.block-container { padding: 2rem 3rem !important; max-width: 1400px; }

/* Hero header */
.hero {
    text-align: center;
    padding: 3rem 0 2rem;
    position: relative;
}
.hero-tag {
    font-family: 'Space Mono', monospace;
    font-size: 0.7rem;
    letter-spacing: 0.3em;
    color: var(--accent);
    text-transform: uppercase;
    margin-bottom: 1rem;
}
.hero-title {
    font-family: 'Space Mono', monospace;
    font-size: 3.2rem;
    font-weight: 700;
    background: linear-gradient(135deg, #00d4ff 0%, #7c3aed 50%, #10b981 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    line-height: 1.1;
    margin: 0 0 1rem;
}
.hero-sub {
    font-size: 1rem;
    color: var(--muted);
    font-weight: 300;
    letter-spacing: 0.02em;
}
.hero-line {
    width: 60px;
    height: 2px;
    background: linear-gradient(90deg, var(--accent), var(--accent2));
    margin: 1.5rem auto 0;
    border-radius: 2px;
}

/* Upload zone */
.upload-section {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 16px;
    padding: 2.5rem;
    margin: 1.5rem 0;
    position: relative;
    overflow: hidden;
}
.upload-section::before {
    content: '';
    position: absolute;
    top: 0; left: 0; right: 0;
    height: 2px;
    background: linear-gradient(90deg, var(--accent), var(--accent2), transparent);
}

/* Stat cards */
.stat-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 1rem;
    margin: 1.5rem 0;
}
.stat-card {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 12px;
    padding: 1.5rem;
    position: relative;
    overflow: hidden;
    transition: transform 0.2s;
}
.stat-card:hover { transform: translateY(-2px); }
.stat-card::after {
    content: '';
    position: absolute;
    bottom: 0; left: 0; right: 0;
    height: 2px;
}
.stat-card.green::after { background: var(--green); }
.stat-card.blue::after { background: var(--accent); }
.stat-card.orange::after { background: var(--orange); }
.stat-label {
    font-family: 'Space Mono', monospace;
    font-size: 0.65rem;
    letter-spacing: 0.15em;
    color: var(--muted);
    text-transform: uppercase;
    margin-bottom: 0.5rem;
}
.stat-value {
    font-family: 'Space Mono', monospace;
    font-size: 2rem;
    font-weight: 700;
    line-height: 1;
    margin-bottom: 0.3rem;
}
.stat-card.green .stat-value { color: var(--green); }
.stat-card.blue .stat-value { color: var(--accent); }
.stat-card.orange .stat-value { color: var(--orange); }
.stat-meta { font-size: 0.8rem; color: var(--muted); }

/* Section headers */
.section-header {
    font-family: 'Space Mono', monospace;
    font-size: 0.7rem;
    letter-spacing: 0.25em;
    color: var(--accent);
    text-transform: uppercase;
    margin: 2.5rem 0 1rem;
    display: flex;
    align-items: center;
    gap: 1rem;
}
.section-header::after {
    content: '';
    flex: 1;
    height: 1px;
    background: var(--border);
}

/* Result image container */
.result-container {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 16px;
    padding: 1.5rem;
    margin: 1rem 0;
}

/* Stagger animations */
@keyframes fadeUp {
    from { opacity: 0; transform: translateY(20px); }
    to { opacity: 1; transform: translateY(0); }
}
.animate { animation: fadeUp 0.5s ease forwards; }

/* Override Streamlit button */
.stButton > button {
    background: linear-gradient(135deg, #00d4ff, #7c3aed) !important;
    color: white !important;
    border: none !important;
    border-radius: 10px !important;
    font-family: 'Space Mono', monospace !important;
    font-size: 0.85rem !important;
    letter-spacing: 0.05em !important;
    padding: 0.75rem 2.5rem !important;
    font-weight: 700 !important;
    transition: opacity 0.2s !important;
    box-shadow: 0 0 20px rgba(0, 212, 255, 0.2) !important;
}
.stButton > button:hover {
    opacity: 0.85 !important;
    box-shadow: 0 0 30px rgba(0, 212, 255, 0.35) !important;
}

/* Tabs */
.stTabs [data-baseweb="tab-list"] {
    background: var(--surface) !important;
    border-radius: 10px;
    padding: 4px;
    gap: 4px;
    border: 1px solid var(--border);
}
.stTabs [data-baseweb="tab"] {
    font-family: 'Space Mono', monospace !important;
    font-size: 0.75rem !important;
    letter-spacing: 0.1em !important;
    color: var(--muted) !important;
    border-radius: 7px !important;
    padding: 0.5rem 1.2rem !important;
}
.stTabs [aria-selected="true"] {
    background: linear-gradient(135deg, rgba(0,212,255,0.15), rgba(124,58,237,0.15)) !important;
    color: var(--accent) !important;
}
.stTabs [data-baseweb="tab-panel"] {
    padding-top: 1.5rem !important;
}

/* Plotly charts dark background fix */
.js-plotly-plot { border-radius: 12px; overflow: hidden; }

/* File uploader */
[data-testid="stFileUploader"] {
    background: var(--surface2) !important;
    border: 1.5px dashed var(--border) !important;
    border-radius: 12px !important;
}

/* Divider */
hr { border-color: var(--border) !important; }

/* Spinner */
.stSpinner > div { border-top-color: var(--accent) !important; }
</style>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────
CLASS_COLORS = {
    0: [0, 0, 0],
    1: [255, 0, 0],
    2: [255, 255, 0],
    3: [0, 0, 255],
    4: [128, 64, 0],
    5: [0, 255, 0],
    6: [255, 165, 0],
}
CLASS_NAMES = {
    0: 'Background',
    1: 'Building',
    2: 'Road',
    3: 'Water',
    4: 'Barren',
    5: 'Forest',
    6: 'Agriculture',
}
PLOTLY_COLORS = {
    0: '#1a1a1a',
    1: '#ef4444',
    2: '#eab308',
    3: '#3b82f6',
    4: '#78350f',
    5: '#22c55e',
    6: '#f97316',
}

# ─────────────────────────────────────────────
# Model loading
# ─────────────────────────────────────────────
@st.cache_resource
def load_model():
    from ultralytics import YOLO
    return YOLO("best.pt")

# ─────────────────────────────────────────────
# Core helpers
# ─────────────────────────────────────────────
def add_labels_to_image(img_pil, results, class_names, class_colors):
    img_array = np.array(img_pil)
    if results[0].masks is not None:
        draw = ImageDraw.Draw(img_pil)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 16)
        except:
            font = ImageFont.load_default()
        boxes = results[0].boxes.xyxy.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy().astype(int)
        for box, cls in zip(boxes, classes):
            x1, y1, x2, y2 = box
            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)
            class_name = class_names.get(cls, f'Class {cls}')
            bg_color = tuple(class_colors.get(cls, [255, 255, 255]))
            brightness = sum(bg_color) / 3
            text_color = (0, 0, 0) if brightness > 127 else (255, 255, 255)
            bbox = draw.textbbox((0, 0), class_name, font=font)
            tw = bbox[2] - bbox[0]
            th = bbox[3] - bbox[1]
            p = 4
            draw.rectangle(
                [center_x - tw//2 - p, center_y - th//2 - p,
                 center_x + tw//2 + p, center_y + th//2 + p],
                fill=bg_color, outline=(0, 0, 0), width=2
            )
            draw.text((center_x - tw//2, center_y - th//2), class_name,
                      fill=text_color, font=font)
    return np.array(img_pil)


def run_segmentation(image_path, model, imgsz=640, conf=0.25, alpha=0.6):
    original_img = cv2.imread(str(image_path))
    original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    h, w = original_img.shape[:2]

    results = model.predict(
        source=str(image_path),
        imgsz=imgsz,
        conf=conf,
        save=False,
        verbose=False,
    )

    segmentation_mask = np.zeros((h, w, 3), dtype=np.uint8)
    classes_present = set()
    class_pixel_counts = {}
    total_pixels = h * w

    if results[0].masks is not None:
        masks = results[0].masks.data.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()

        for mask, cls, conf_val in zip(masks, classes, confs):
            classes_present.add(cls)
            mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
            mask_binary = (mask_resized > 0.5).astype(np.uint8)
            pixel_count = int(mask_binary.sum())
            if cls not in class_pixel_counts:
                class_pixel_counts[cls] = {"pixels": 0, "confidences": []}
            class_pixel_counts[cls]["pixels"] += pixel_count
            class_pixel_counts[cls]["confidences"].append(float(conf_val))
            color = CLASS_COLORS.get(cls, [255, 255, 255])
            for c in range(3):
                segmentation_mask[:, :, c] = np.where(
                    mask_binary == 1, color[c], segmentation_mask[:, :, c]
                )

    overlay_img = cv2.addWeighted(original_img, 1 - alpha, segmentation_mask, alpha, 0)
    mask_with_labels = add_labels_to_image(
        Image.fromarray(segmentation_mask), results, CLASS_NAMES, CLASS_COLORS
    )
    overlay_with_labels = add_labels_to_image(
        Image.fromarray(overlay_img), results, CLASS_NAMES, CLASS_COLORS
    )

    # Build per-class stats
    class_stats = {}
    for cls, data in class_pixel_counts.items():
        class_stats[cls] = {
            "name": CLASS_NAMES.get(cls, f"Class {cls}"),
            "pixels": data["pixels"],
            "area_pct": round(data["pixels"] / total_pixels * 100, 2),
            "avg_conf": round(np.mean(data["confidences"]) * 100, 1),
            "instances": len(data["confidences"]),
        }

    return {
        "original": original_img,
        "mask": mask_with_labels,
        "overlay": overlay_with_labels,
        "classes_present": classes_present,
        "class_stats": class_stats,
        "total_pixels": total_pixels,
        "results": results,
    }


# ─────────────────────────────────────────────
# Plotly chart builders
# ─────────────────────────────────────────────
DARK_LAYOUT = dict(
    paper_bgcolor='rgba(17,24,39,0)',
    plot_bgcolor='rgba(17,24,39,0)',
    font=dict(family='DM Sans', color='#94a3b8'),
    margin=dict(l=20, r=20, t=40, b=20),
)

def make_donut_chart(class_stats):
    labels, values, colors = [], [], []
    for cls, s in class_stats.items():
        labels.append(s["name"])
        values.append(s["area_pct"])
        colors.append(PLOTLY_COLORS.get(cls, '#888'))

    fig = go.Figure(go.Pie(
        labels=labels,
        values=values,
        hole=0.6,
        marker=dict(colors=colors, line=dict(color='#0a0e1a', width=2)),
        textinfo='label+percent',
        textfont=dict(size=13, family='DM Sans'),
        hovertemplate='<b>%{label}</b><br>Coverage: %{value:.1f}%<extra></extra>',
    ))
    fig.update_layout(
        **DARK_LAYOUT,
        title=dict(text='Land Cover Distribution', font=dict(size=16, color='#e2e8f0'), x=0.5),
        showlegend=False,
        height=360,
        annotations=[dict(
            text='COVERAGE', x=0.5, y=0.5, font_size=11,
            font_color='#64748b', font_family='Space Mono', showarrow=False
        )]
    )
    return fig


def make_confidence_bar(class_stats):
    names = [s["name"] for s in class_stats.values()]
    confs = [s["avg_conf"] for s in class_stats.values()]
    colors = [PLOTLY_COLORS.get(cls, '#888') for cls in class_stats.keys()]

    fig = go.Figure(go.Bar(
        x=confs,
        y=names,
        orientation='h',
        marker=dict(
            color=colors,
            line=dict(color='rgba(0,0,0,0)', width=0),
        ),
        text=[f'{c:.1f}%' for c in confs],
        textposition='outside',
        textfont=dict(size=13, color='#e2e8f0', family='Space Mono'),
        hovertemplate='<b>%{y}</b><br>Avg Confidence: %{x:.1f}%<extra></extra>',
    ))
    fig.update_layout(
        **DARK_LAYOUT,
        title=dict(text='Detection Confidence by Class', font=dict(size=16, color='#e2e8f0'), x=0.5),
        xaxis=dict(range=[0, 115], showgrid=False, showticklabels=False, zeroline=False),
        yaxis=dict(showgrid=False, tickfont=dict(size=13)),
        height=360,
        bargap=0.35,
    )
    return fig


def make_area_bar(class_stats):
    names = [s["name"] for s in class_stats.values()]
    pixels = [s["pixels"] for s in class_stats.values()]
    pcts = [s["area_pct"] for s in class_stats.values()]
    colors = [PLOTLY_COLORS.get(cls, '#888') for cls in class_stats.keys()]

    fig = go.Figure(go.Bar(
        x=names,
        y=pcts,
        marker=dict(color=colors, line=dict(color='rgba(0,0,0,0)')),
        text=[f'{p}%' for p in pcts],
        textposition='outside',
        textfont=dict(size=12, color='#e2e8f0', family='Space Mono'),
        hovertemplate='<b>%{x}</b><br>Area: %{y:.2f}%<extra></extra>',
    ))
    fig.update_layout(
        **DARK_LAYOUT,
        title=dict(text='Area Coverage per Class', font=dict(size=16, color='#e2e8f0'), x=0.5),
        xaxis=dict(showgrid=False, tickfont=dict(size=13)),
        yaxis=dict(showgrid=True, gridcolor='rgba(30,45,69,0.8)', ticksuffix='%'),
        height=360,
        bargap=0.4,
    )
    return fig


# ─────────────────────────────────────────────
# UI
# ─────────────────────────────────────────────
st.markdown("""
<div class="hero animate">
    <div class="hero-tag">🛰️ Remote Sensing · AI Segmentation</div>
    <div class="hero-title">Urban analysis AI</div>
    <div class="hero-sub">Satellite image land-cover analysis powered by YOLOv26</div>
    <div class="hero-line"></div>
</div>
""", unsafe_allow_html=True)

# Upload section
st.markdown('<div class="section-header">Upload</div>', unsafe_allow_html=True)

col_upload, col_preview = st.columns([1, 1], gap="large")

with col_upload:
    uploaded_file = st.file_uploader(
        "Drop a satellite image here",
        type=["jpg", "jpeg", "png", "tif", "tiff"],
        label_visibility="collapsed",
    )
    if uploaded_file:
        st.caption(f"📁 **{uploaded_file.name}** · {uploaded_file.size / 1024:.1f} KB")

        conf_thresh = st.slider("Confidence Threshold", 0.1, 0.9, 0.25, 0.05)
        alpha_val = st.slider("Overlay Opacity", 0.1, 0.9, 0.6, 0.05)

        run_btn = st.button("⬡  Run Analysis", use_container_width=True)

with col_preview:
    if uploaded_file:
        img = Image.open(uploaded_file)
        st.image(img, caption="Preview", use_container_width=True)
    else:
        st.markdown("""
        <div style="height:260px;display:flex;align-items:center;justify-content:center;
                    background:var(--surface);border-radius:12px;border:1px solid var(--border);
                    color:var(--muted);font-family:'Space Mono',monospace;font-size:0.75rem;
                    letter-spacing:0.1em;">
            NO IMAGE LOADED
        </div>
        """, unsafe_allow_html=True)

# ─────────────────────────────────────────────
# Analysis
# ─────────────────────────────────────────────
if uploaded_file and run_btn:
    model = load_model()

    with tempfile.NamedTemporaryFile(delete=False, suffix=Path(uploaded_file.name).suffix) as tmp:
        tmp.write(uploaded_file.getvalue())
        tmp_path = tmp.name

    with st.spinner("Running segmentation model…"):
        data = run_segmentation(tmp_path, model, conf=conf_thresh, alpha=alpha_val)

    os.unlink(tmp_path)
    cs = data["class_stats"]

    # ── Summary cards ──────────────────────────────────────────────
    if cs:
        dominant = max(cs.values(), key=lambda x: x["pixels"])
        green_pct = sum(
            s["area_pct"] for cls, s in cs.items() if cls in [5, 6]
        )
        urban_pct = sum(
            s["area_pct"] for cls, s in cs.items() if cls in [1, 2]
        )
        total_instances = sum(s["instances"] for s in cs.values())

        st.markdown('<div class="section-header">Summary</div>', unsafe_allow_html=True)
        st.markdown(f"""
        <div class="stat-grid">
            <div class="stat-card green">
                <div class="stat-label">Dominant Cover</div>
                <div class="stat-value" style="font-size:1.4rem;">{dominant["name"]}</div>
                <div class="stat-meta">{dominant["area_pct"]}% of image area</div>
            </div>
            <div class="stat-card blue">
                <div class="stat-label">Green Index</div>
                <div class="stat-value">{green_pct:.1f}%</div>
                <div class="stat-meta">Forest + Agriculture</div>
            </div>
            <div class="stat-card orange">
                <div class="stat-label">Urban Index</div>
                <div class="stat-value">{urban_pct:.1f}%</div>
                <div class="stat-meta">Buildings + Roads</div>
            </div>
        </div>
        """, unsafe_allow_html=True)

    # ── Tabs ───────────────────────────────────────────────────────
    st.markdown('<div class="section-header">Results</div>', unsafe_allow_html=True)
    tab1, tab2, tab3 = st.tabs(["🖼️  Visualizations", "📊  Charts", "🔬  Class Details"])

    with tab1:
        c1, c2, c3 = st.columns(3)
        with c1:
            st.markdown('<p style="font-family:\'Space Mono\',monospace;font-size:0.7rem;letter-spacing:0.15em;color:#64748b;text-transform:uppercase;">Original</p>', unsafe_allow_html=True)
            st.image(data["original"], use_container_width=True)
        with c2:
            st.markdown('<p style="font-family:\'Space Mono\',monospace;font-size:0.7rem;letter-spacing:0.15em;color:#64748b;text-transform:uppercase;">Segmentation Mask</p>', unsafe_allow_html=True)
            st.image(data["mask"], use_container_width=True)
        with c3:
            st.markdown('<p style="font-family:\'Space Mono\',monospace;font-size:0.7rem;letter-spacing:0.15em;color:#64748b;text-transform:uppercase;">Overlay</p>', unsafe_allow_html=True)
            st.image(data["overlay"], use_container_width=True)

    with tab2:
        if cs:
            left, right = st.columns(2, gap="large")
            with left:
                st.plotly_chart(make_donut_chart(cs), use_container_width=True, config={"displayModeBar": False})
                st.plotly_chart(make_area_bar(cs), use_container_width=True, config={"displayModeBar": False})
            with right:
                st.plotly_chart(make_confidence_bar(cs), use_container_width=True, config={"displayModeBar": False})
        else:
            st.info("No detections found.")

    with tab3:
        if cs:
            for cls, s in sorted(cs.items(), key=lambda x: -x[1]["area_pct"]):
                hex_color = PLOTLY_COLORS.get(cls, '#888')
                st.markdown(f"""
                <div style="background:var(--surface);border:1px solid var(--border);
                            border-radius:12px;padding:1.2rem 1.5rem;margin-bottom:0.8rem;
                            border-left:4px solid {hex_color};">
                    <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:1rem;">
                        <div>
                            <span style="font-family:'Space Mono',monospace;font-size:1rem;
                                         font-weight:700;color:{hex_color};">{s["name"]}</span>
                            <span style="font-size:0.75rem;color:var(--muted);margin-left:1rem;">
                                Class {cls}
                            </span>
                        </div>
                        <div style="display:flex;gap:2rem;">
                            <div style="text-align:center;">
                                <div style="font-family:'Space Mono',monospace;font-size:1.3rem;
                                             font-weight:700;color:{hex_color};">{s["area_pct"]}%</div>
                                <div style="font-size:0.7rem;color:var(--muted);">Area</div>
                            </div>
                            <div style="text-align:center;">
                                <div style="font-family:'Space Mono',monospace;font-size:1.3rem;
                                             font-weight:700;color:#e2e8f0;">{s["avg_conf"]}%</div>
                                <div style="font-size:0.7rem;color:var(--muted);">Confidence</div>
                            </div>
                            <div style="text-align:center;">
                                <div style="font-family:'Space Mono',monospace;font-size:1.3rem;
                                             font-weight:700;color:#e2e8f0;">{s["instances"]}</div>
                                <div style="font-size:0.7rem;color:var(--muted);">Instances</div>
                            </div>
                            <div style="text-align:center;">
                                <div style="font-family:'Space Mono',monospace;font-size:1.3rem;
                                             font-weight:700;color:#e2e8f0;">{s["pixels"]:,}</div>
                                <div style="font-size:0.7rem;color:var(--muted);">Pixels</div>
                            </div>
                        </div>
                    </div>
                </div>
                """, unsafe_allow_html=True)
        else:
            st.info("No class statistics available.")

elif not uploaded_file:
    st.markdown("""
    <div style="text-align:center;padding:4rem 2rem;color:var(--muted);">
        <div style="font-size:3rem;margin-bottom:1rem;">🛰️</div>
        <div style="font-family:'Space Mono',monospace;font-size:0.8rem;
                    letter-spacing:0.2em;text-transform:uppercase;">
            Upload a satellite image to begin analysis
        </div>
    </div>
    """, unsafe_allow_html=True)

Writing app.py


In [8]:
from pyngrok import ngrok

ngrok_key = os.environ.get("Ngrok_API_KEY")
port = 8501

ngrok.set_auth_token(ngrok_key)
ngrok.connect(port).public_url

'https://undeliriously-unvigorous-grayson.ngrok-free.dev'

In [ ]:
!rm -rf logs.txt && streamlit run app.py &>/content/logs.txt